# 1. Data Acquisition

In [105]:
'''
Load the dataset from Hugging Face 

from datasets import load_dataset
df = load_dataset("divarofficial/real_estate_ads")

or download it
'''

import pandas as pd

df = pd.read_csv("real_estate_ads.csv")
df.head()


C:\Users\Amir\AppData\Local\Temp\ipykernel_13736\2357479437.py:12: DtypeWarning: Columns (10,26,28,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("real_estate_ads.csv")


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,...,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,...,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Data Understandig

In [106]:
print(f"Dataset Shape: {df.shape}")


Dataset Shape: (1000000, 60)


In [107]:
# Save the dataset information to a text file because df.info() prints does not represent the full information of the dataset. It truncates the output if the dataset is too large.
from io import StringIO
buffer = StringIO()
df.info(buf=buffer)
with open("df_info.txt", "w", encoding="utf-8") as f:
    f.write(buffer.getvalue())


In [108]:
print(f"Duplicated Rows Count: {df.duplicated().sum()} \n")

print(df["cat2_slug"].value_counts())

Duplicated Rows Count: 0 

residential-sell        558708
residential-rent        276558
commercial-rent          76567
commercial-sell          38861
temporary-rent           29903
real-estate-services     19403
Name: cat2_slug, dtype: int64


# 3. Data Cleaning

In [109]:
'''
The purpose of this project is to predict the price of a house based on its features.
Therefore, first we will keep only the residential sell category and remove all other categories from the dataset.
'''

df = df[df["cat2_slug"] == "residential-sell"]

In [110]:
df["cat3_slug"].value_counts()

apartment-sell      303385
plot-old            133570
house-villa-sell    121753
Name: cat3_slug, dtype: int64

After inspecting the cat3_slug column, we can see that some fraction of the data is related to "land" category.
Since we are only interested in predicting the price of houses, we will remove this category from the dataset.

We can use this data to predict the price of land in a separate project.

In [111]:
df = df[df["cat3_slug"] != "plot-old"]

In [112]:
#Drop the columns that are not useful for our analysis
drop_columns = [
    "cat2_slug", "user_type", "description", "title", "rent_mode",
    "rent_value", "rent_to_single", "price_mode","floor_material",
    "rent_type", "credit_mode", "credit_value", "rent_credit_transform",
    "transformable_price", "transformable_credit", "transformed_credit",
    "transformable_rent", "transformed_rent", "has_business_deed",
    "regular_person_capacity", "extra_person_capacity", "is_rebuilt",
    "cost_per_extra_person", "rent_price_on_regular_days",
    "rent_price_on_special_days", "rent_price_at_weekends",
    "location_latitude", "location_longitude", "location_radius",
    "has_warm_water_provider", "has_balcony", "has_heating_system",
    "has_cooling_system", "has_security_guard", "has_barbecue",
    "has_restroom", "building_direction"
]

df.drop(columns=drop_columns, inplace=True)


## 3.1 Remove Or Modify Invalid Values

In [113]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 425138 entries, 1 to 999997
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   cat3_slug           425138 non-null  object 
 1   city_slug           425137 non-null  object 
 2   neighborhood_slug   197915 non-null  object 
 3   created_at_month    425138 non-null  object 
 4   price_value         425128 non-null  float64
 5   land_size           121735 non-null  float64
 6   building_size       425118 non-null  float64
 7   deed_type           253455 non-null  object 
 8   floor               303377 non-null  object 
 9   rooms_count         425117 non-null  object 
 10  total_floors_count  187087 non-null  object 
 11  unit_per_floor      184934 non-null  object 
 12  has_elevator        303378 non-null  object 
 13  has_warehouse       425116 non-null  object 
 14  has_parking         425117 non-null  object 
 15  construction_year   425118 non-nul

In [114]:
'''
Price value should be a numeric value and not null and zero
'''
print(f"Number of rows with null and zero price values: { df[df['price_value'].isnull()].shape[0] } + { df[df['price_value'] == 0].shape[0] }")

df = df[df["price_value"] != 0]
df = df[df["price_value"].notnull()] 

Number of rows with null and zero price values: 10 + 1703


In [115]:
print(f"Number of rows with zero and null land size values: { df[df['land_size'] == 0].shape[0] } + { df[df['land_size'].isnull()].shape[0] }")
print(f"Number of rows with zero and null building size values: { df[df['building_size'] == 0].shape[0] } + { df[df['building_size'].isnull()].shape[0] }")


Number of rows with zero and null land size values: 0 + 302655
Number of rows with zero and null building size values: 0 + 13


We will work on with different features and dimentions, thus for now we don't remove any rows with zero or null land and building size values.

In [116]:
'''
floor values should be numeric and integer. 
note: floor values can be negative, for example -1 means the floor below the ground floor.
'''
df["floor"].value_counts()



2       72323
3       64216
1       63504
4       43285
5       22614
0       14713
6        8490
7        3380
8        1890
10       1420
9        1303
11        931
-1        831
12        795
13        528
14        520
30+       419
15        270
16        178
17        154
18        116
20        116
19         88
21         65
25         65
22         49
23         46
2.0        45
1.0        44
30         42
3.0        35
24         29
4.0        25
28         24
27         17
5.0        14
26         14
29         12
0.0         9
7.0         5
6.0         5
20.0        1
18.0        1
8.0         1
22.0        1
14.0        1
9.0         1
11.0        1
10.0        1
Name: floor, dtype: int64

After investigting the floor values, we relize that some values are in float type and some of them are string(like "-1").
There is a "30+" value that refers to units located on the above 30th floor and by default we replace it with 31. However we are going to compare `total_floors_count` value and `floor` value and if `floor` value will be more than `total_floors_count`, it will be detect as a invalid value and it will be assigned into null.
Note that we won't remove invalid or null values from `floor` feature.

In [117]:
# convert each floor and total floors count value to int, handling float values and string values like "30+" -> 31
df["floor"] = df["floor"].astype(str).str.strip().str.replace(r"^30\+$", "31", regex=True)
df["floor"] = pd.to_numeric(df["floor"], errors="coerce").astype("Int32")
df["total_floors_count"] = pd.to_numeric(df["total_floors_count"], errors="coerce").astype("Int32")

print("floor values:")
print(df["floor"].unique())
print("total floors count values:")
print(df["total_floors_count"].unique())


floor values:
<IntegerArray>
[   3,    4,    2, <NA>,    5,    1,    0,   30,    7,    8,    6,   12,   -1,
   13,   19,   16,   11,   31,   10,    9,   15,   22,   18,   14,   17,   20,
   23,   24,   21,   25,   28,   29,   27,   26]
Length: 34, dtype: Int32
total floors count values:
<IntegerArray>
[<NA>,    6,   14,    4,    3,    5,    7,    8,   11,    2,   12,   17,    9,
   25,   10,   15,   21,   20,   22,   13,   19,   24,   16,   26,   18,   30,
   29,   23,   27,   28]
Length: 30, dtype: Int32


In [118]:
# check the validation of floor number based on total floors count 

mask = df["floor"].notna() & df["total_floors_count"].notna() & (df["floor"] > df["total_floors_count"])
df.loc[mask, "floor"] = pd.NA
df["floor"] = df["floor"].astype("Int32")

In [119]:
df["rooms_count"].value_counts()

دو              232360
سه               82828
یک               82475
چهار             13423
پنج یا بیشتر      6870
بدون اتاق         5455
Name: rooms_count, dtype: int64

In [120]:
# convert each rooms_count values into numeric, handling string values like "بدون اتاق " to 0 and "پنج یا بیشتر" to 5

rooms_map = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5,
}

rooms = df["rooms_count"].str.strip()
df["rooms_count"] = pd.to_numeric(rooms.replace(rooms_map), errors="coerce").astype("Int32")
df["rooms_count"].value_counts()

2    232360
3     82828
1     82475
4     13423
5      6870
0      5455
Name: rooms_count, dtype: Int64

In [121]:
df["unit_per_floor"].value_counts()

2              74915
1              53912
4              24631
3              20450
5               3105
6               3089
8               2280
more_than_8     1431
7                584
unselect         149
Name: unit_per_floor, dtype: int64

In [122]:
# convert each unit_per_floor value to numeric, handling string values like "more_than_8" -> 9

df["unit_per_floor"] = df["unit_per_floor"].astype(str).str.strip().str.replace(r"^more_than_8$", "9", regex=True)
df["unit_per_floor"] = pd.to_numeric(df["unit_per_floor"], errors="coerce").astype("Int32")
df["unit_per_floor"].value_counts()


2    74915
1    53912
4    24631
3    20450
5     3105
6     3089
8     2280
9     1431
7      584
Name: unit_per_floor, dtype: Int64

In [123]:
df["construction_year"].value_counts()

۱۴۰۳           80316
۱۴۰۲           38036
۱۴۰۰           23488
۱۳۹۰           22633
۱۳۹۵           22594
۱۴۰۱           19456
۱۳۹۷           17748
۱۳۹۶           17367
۱۳۹۸           17366
۱۳۸۵           15758
۱۳۹۳           15225
۱۳۹۴           13951
۱۳۹۹           13730
۱۳۹۲           13121
۱۳۸۸           11981
۱۳۸۰            9582
قبل از ۱۳۷۰     8889
۱۳۹۱            8525
۱۳۸۹            8379
۱۳۸۷            7528
۱۳۸۶            7477
۱۳۸۳            5790
۱۳۸۴            5089
۱۳۸۲            4022
۱۳۷۵            3343
۱۳۷۱            2510
۱۳۸۱            2161
۱۳۷۸            1659
۱۳۷۹            1282
۱۳۷۷            1092
۱۳۷۲             929
۱۳۷۳             904
۱۳۷۶             849
۱۳۷۴             632
Name: construction_year, dtype: int64

In [124]:
# convert each construction_year value to numeric, handling string values like "قبل از 1370" to 1369

years_map = {
        '۱۳۸۴': 1384,
        '۱۴۰۳': 1403,
        '۱۳۹۳': 1393,
        '۱۳۹۶': 1396,
        '۱۳۸۷': 1387,
        '۱۴۰۲': 1402,
        '۱۳۸۸': 1388,
        '۱۳۹۸': 1398,
        '۱۴۰۰': 1400,
        '۱۳۹۲': 1392,
        '۱۴۰۱': 1401,
        '۱۳۹۴': 1394,
        '۱۳۸۳': 1383,
        '۱۳۹۹': 1399,
        '۱۳۹۰': 1390,
        '۱۳۸۵': 1385,
        '۱۳۸۲': 1382,
        '۱۳۹۵': 1395,
        '۱۳۹۱': 1391,
        '۱۳۸۶': 1386,
        '۱۳۸۰': 1390,
        '۱۳۷۵': 1375,
        '۱۳۹۷': 1397,
        '۱۳۷۲': 1372,
        '۱۳۷۴': 1374,
        'قبل از ۱۳۷۰': 1369,
        '۱۳۸۹': 1389,
        '۱۳۷۹': 1379,
        '۱۳۷۱': 1371,
        '۱۳۸۱': 1381,
        '۱۳۷۳': 1373,
        '۱۳۷۸': 1378,
        '۱۳۷۷': 1377,
        '۱۳۷۶': 1376
    
}


years = df["construction_year"].str.strip()
df["construction_year"] = pd.to_numeric(years.replace(years_map), errors="coerce").astype("Int32")

df["construction_year"].value_counts()


1403    80316
1402    38036
1390    32215
1400    23488
1395    22594
1401    19456
1397    17748
1396    17367
1398    17366
1385    15758
1393    15225
1394    13951
1399    13730
1392    13121
1388    11981
1369     8889
1391     8525
1389     8379
1387     7528
1386     7477
1383     5790
1384     5089
1382     4022
1375     3343
1371     2510
1381     2161
1378     1659
1379     1282
1377     1092
1372      929
1373      904
1376      849
1374      632
Name: construction_year, dtype: Int64

Features starting with **has** refer to the home's amenities; Like: `has_gas`, `has_water`, `has_elevator` and etc. We won't directly work with these features but some of them have a significant impact and we should consider it.

In [125]:
for col in df.columns:
    if col.startswith("has_"):
        print(f"{col} values: {df[col].unique()} - type: {df[col].dtype}")

has_elevator values: [True False nan] - type: object
has_warehouse values: [True False nan] - type: object
has_parking values: [True False nan] - type: object
has_water values: [nan False True] - type: object
has_electricity values: [nan False True] - type: object
has_gas values: [nan False True] - type: object
has_pool values: [nan False True] - type: object
has_jacuzzi values: [nan False True] - type: object
has_sauna values: [nan False True] - type: object


In [126]:
# convert all the has_* columns to boolean values

for col in df.columns:
    if col.startswith("has_"):
        df[col] = df[col].astype("boolean")
        print(f"{col} values: {df[col].unique()} - type: {df[col].dtype}")

has_elevator values: <BooleanArray>
[True, False, <NA>]
Length: 3, dtype: boolean - type: boolean
has_warehouse values: <BooleanArray>
[True, False, <NA>]
Length: 3, dtype: boolean - type: boolean
has_parking values: <BooleanArray>
[True, False, <NA>]
Length: 3, dtype: boolean - type: boolean
has_water values: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean - type: boolean
has_electricity values: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean - type: boolean
has_gas values: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean - type: boolean
has_pool values: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean - type: boolean
has_jacuzzi values: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean - type: boolean
has_sauna values: <BooleanArray>
[<NA>, False, True]
Length: 3, dtype: boolean - type: boolean


In [127]:
df["property_type"].value_counts()

jungle             15040
other               2917
beach               2711
jungle-mountain     1801
mountain             150
Name: property_type, dtype: int64

`property_type` column doesn't need to modify or type convertion

In [128]:
df["created_at_month"].value_counts()

2024-11-01 00:00:00    55648
2024-12-01 00:00:00    54888
2024-10-01 00:00:00    54251
2024-07-01 00:00:00    52216
2024-08-01 00:00:00    51355
2024-06-01 00:00:00    49732
2024-05-01 00:00:00    49528
2024-09-01 00:00:00    47867
2024-04-01 00:00:00     4023
2024-03-01 00:00:00      996
2025-01-01 00:00:00      748
2024-02-01 00:00:00      691
2024-01-01 00:00:00      542
2023-12-01 00:00:00      291
2023-11-01 00:00:00      182
2025-02-01 00:00:00      110
2023-10-01 00:00:00       96
2023-09-01 00:00:00       75
2023-08-01 00:00:00       47
2023-07-01 00:00:00       44
2023-06-01 00:00:00       29
2023-05-01 00:00:00       22
2023-04-01 00:00:00        7
2022-11-01 00:00:00        5
2023-01-01 00:00:00        5
2022-12-01 00:00:00        4
2022-08-01 00:00:00        4
2023-02-01 00:00:00        4
2025-03-01 00:00:00        4
2022-10-01 00:00:00        3
2023-03-01 00:00:00        3
2022-06-01 00:00:00        1
2022-07-01 00:00:00        1
2021-06-01 00:00:00        1
2021-02-01 00:

To reduce the space that `created_at_month` property occupied, we just store **the last two digit of year** and **month**. For example: '2024-05-01 00:00:00' -> 2405.

So the object format will be converted to Int.

In [129]:
# convert the created_at_month values to numeric, handling string values like "2024-05-01 00:00:00" to 2405.

df["created_at_month"] = df["created_at_month"].str.replace("-", "").str.slice(2, 6).astype("Int32")
df["created_at_month"].value_counts()

2411    55648
2412    54888
2410    54251
2407    52216
2408    51355
2406    49732
2405    49528
2409    47867
2404     4023
2403      996
2501      748
2402      691
2401      542
2312      291
2311      182
2502      110
2310       96
2309       75
2308       47
2307       44
2306       29
2305       22
2304        7
2211        5
2301        5
2212        4
2208        4
2302        4
2503        4
2210        3
2303        3
2206        1
2207        1
2106        1
2102        1
2205        1
Name: created_at_month, dtype: Int64

In [130]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 423425 entries, 1 to 999997
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   cat3_slug           423425 non-null  object 
 1   city_slug           423424 non-null  object 
 2   neighborhood_slug   197565 non-null  object 
 3   created_at_month    423425 non-null  Int32  
 4   price_value         423425 non-null  float64
 5   land_size           120770 non-null  float64
 6   building_size       423412 non-null  float64
 7   deed_type           252595 non-null  object 
 8   floor               301900 non-null  Int32  
 9   rooms_count         423411 non-null  Int32  
 10  total_floors_count  186469 non-null  Int32  
 11  unit_per_floor      184397 non-null  Int32  
 12  has_elevator        302637 non-null  boolean
 13  has_warehouse       423411 non-null  boolean
 14  has_parking         423411 non-null  boolean
 15  construction_year   423412 non-nul

In [131]:
# save the cleaned dataset to a new CSV file
df.to_csv("cleaned_real_estate_ads.csv", index=False)